In [1]:
import os
import sys

# Standard interactive replacement for the 'parent directory' hack
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from PPRCalculator import PPRCalculator
from ModelData import ModelData

In [61]:
def apply_ecopath_defaults(df, DC, det_fate=None, zero_catch=False, zero_biomass_accum=False, default_gs=False):
    """
    Applies Ecopath defaults and ensures flows are synced with ratios.
    Assumes p, q, ee, and catch are given. assumes p, q not given for detritus group.
        calculates M0 = p*(1-ee)
    Assumes gs is 0.2 for regular groups and 0 for pp, det, diet_import.
        calculates egestion = q*gs.
    Assumes biomas_accum, net_migration are 0 where currently nan.
    Calculates once cell missing from p, q, egestion, respiration.
    Calculates predation = Z.sum(axis=1).
        Calculates once cell missing from M0, predation, catch, biomass_accum, net_migration.
    """
    df = df.sort_index(ascending=False)
    is_regular = df["trophic_info"] == "Regular"
    is_import = df["trophic_info"] == "Import"
    is_det = df["trophic_info"] == "DET"
    is_pp = df["trophic_info"] == "PP"

    # original_nas = df.isna()

    # cols with constant value of 0 or 1:
    cols_where_default_is_zero = ['detritus_import', 'immigration', 'emigration', 'net_migration']
    if zero_catch:
        cols_where_default_is_zero.append('catch')
    if zero_biomass_accum:
        cols_where_default_is_zero.append('biomass_accum')
    fill_dict = {col: 0 for col in cols_where_default_is_zero}
    df = df.fillna(fill_dict)
    df.loc[:, 'biomass'] = df['biomass'].fillna(1)

    # get Z which is correct for all regular groups:
    Z = DC.mul(df['q'].fillna(0), axis='index')
    Z.loc[is_det | is_pp | is_import, :] = 0
    Z = Z.sort_index(ascending=False).sort_index(ascending=False, axis=1)

    # set default values for non-regular groups:
    df.loc[is_pp | is_import | is_det, ['egestion', 'respiration']] = 0
    df.loc[is_import | is_det, 'M0'] = 0
    df.loc[is_import | is_det, 'ee'] = 1

    # fillna for gs and egestion:
    df.loc[~is_regular, 'gs'] = df.loc[~is_regular, 'gs'].fillna(0)
    if default_gs:
        df.loc[is_regular, 'gs'] = df.loc[is_regular, 'gs'].fillna(0.2)

    # Sync gs -> egestion
    mask_sync = df['egestion'].isna() & df['q'].notna() & df['gs'].notna()
    df.loc[mask_sync, 'egestion'] = df['q'] * df['gs']

    # Sync ee -> M0
    mask_sync = df['M0'].isna() & df["p"].notna() & df["ee"].notna()
    df.loc[mask_sync, 'M0'] = df["p"] * (1 - df["ee"])

    # q for detritus:
    df['flow_to_det'] = df['M0'] + df['egestion']
    if det_fate is not None:
        detritus_inflows = det_fate.mul(df['flow_to_det'], axis='index').sum(axis=0)
        df.loc[is_det, 'q'] = df.loc[is_det, 'q'].fillna(detritus_inflows)
    else:
        df.loc[is_det, 'q'] = df.loc[is_det, 'q'].fillna(df['M0'].sum() + df['egestion'].sum())  # set q of detritus as M0.sum() + egestion.sum()

    # predation:
    df.loc[:, 'predation'] = Z.sum(axis=0)

    # for non-regular groups, egestion and respiration are nan so q = p:
    df.loc[is_det, 'p'] = df['q']
    df.loc[is_pp | is_import, 'q'] = df['p']
    
    # pb and qb:
    df.loc[:, 'pb'] = (df['p'] / df['biomass']).fillna(0)
    df.loc[:, 'qb'] = (df['q'] / df['biomass']).fillna(0)

    def _solve_linear_equation(df, eq_cols, signs):
        # Identify rows where exactly ONE variable is missing (otherwise it's unsolvable this way)
        solvable_mask = df[eq_cols].isna().sum(axis=1) == 1

        # Calculate the net sum of all known terms (Pandas ignores NaNs by default in .sum)
        balance_sum = (df[eq_cols] * signs).sum(axis=1, skipna=True)

        # Fill the missing cells
        for col in eq_cols:
            # Target rows where THIS column is the missing one, and the equation is solvable
            target_cells = df[col].isna() & solvable_mask
            
            # The missing value is the negative balance_sum divided by the column's sign
            df.loc[target_cells, col] = -balance_sum[target_cells] / signs[col]
        
        return df
    
    # consumption equation:
    eq_cols = ['q', 'p', 'respiration', 'egestion']
    signs = pd.Series({
        'q': 1, 
        'p': -1, 
        'respiration': -1, 
        'egestion': -1, 
    })
    df = _solve_linear_equation(df, eq_cols, signs)
    
    # production equation:
    eq_cols = ['p', 'M0', 'catch', 'predation', 'net_migration', 'biomass_accum']
    signs = pd.Series({
        'p': 1, 
        'M0': -1, 
        'catch': -1, 
        'predation': -1, 
        'net_migration': -1, 
        'biomass_accum': -1
    })
    df = _solve_linear_equation(df, eq_cols, signs)

    # finalize ratios:
    df.loc[is_regular, 'ee'] = 1 - (df['M0'] / df['p'])
    df.loc[is_regular, 'gs'] = (df['egestion'] / df['q'])
    df['ge'] = (df['p'] / df['q']).fillna(0)
    df['flow_to_det'] = df['flow_to_det'].fillna(df['M0'] + df['egestion'])

    return df

In [62]:
model = PPRCalculator(116)
model.get_groups_df().round(5)[['group_name', 'trophic_info', 'tl', 'ee', 'p', 'q', 'biomass_accum', 'net_migration', 'M0', 'predation', 'catch', 'gs', 'egestion', 'respiration']]
model.get_groups_df().round(5)

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
33,diet_import,Import,1.00000,1.00000,1.00000,0.00000,NaN,NaN,NaN,0.00000,...,0.00000,0.000,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,NaN
32,Detritus,DET,1.00000,1.00000,0.99965,0.00000,132.61000,NaN,NaN,1283.66658,...,0.00000,0.000,0.00000,0.00000,512.76917,0.0,0.0,0.0,NaN,0.0
31,Phytoplankton,PP,1.00000,1.00000,0.85716,0.00000,12.95000,105.03494,0.00000,1360.20250,...,194.28466,0.000,0.00000,0.00000,0.00000,0.0,0.0,0.0,194.28466,0.0
30,Small zooplankton,Regular,2.10254,0.25000,0.79477,0.00000,71.86000,4.81501,19.26006,346.00691,...,71.01249,0.455,629.73255,408.28815,0.00000,0.0,0.0,0.0,700.74504,0.0
29,Large zooplankton,Regular,2.41896,0.13174,0.54362,0.00000,14.68000,2.62915,19.95700,38.59588,...,17.61443,0.500,146.48436,107.88848,0.00000,0.0,0.0,0.0,164.09879,0.0
28,Other ben. inver.,Regular,2.00000,0.30000,0.32719,0.00000,5.70000,1.60936,5.36454,9.17336,...,6.17197,0.300,9.17336,12.23115,0.00000,0.0,0.0,0.0,15.34533,0.0
27,Polychaetes,Regular,2.15473,0.27396,0.54807,0.00000,14.82000,2.11052,7.70368,31.27794,...,14.13548,0.100,11.41685,71.47374,0.00000,0.0,0.0,0.0,25.55233,0.0
26,Molluscs,Regular,2.00000,0.09000,0.02180,0.03450,53.44000,0.42948,4.77202,22.95153,...,22.45119,0.300,76.50509,155.56035,0.00000,0.0,0.0,0.0,98.95628,0.0
25,Echinoderms,Regular,2.00000,0.30000,0.02321,0.00000,81.90000,0.41024,1.36747,33.59866,...,32.81892,0.300,33.59866,44.79822,0.00000,0.0,0.0,0.0,66.41758,0.0


In [66]:
groups_df = model.get_groups_df().copy()
# groups_df[['egestion', 'respiration', 'biomass_accum', 'emigration', 'immigration', 'net_migration', 'M0']] = np.nan
groups_df[['M0', 'ee', 'respiration', 'egestion', 'biomass_accum', 'gs']] = np.nan

apply_ecopath_defaults(groups_df, model.get_DC(), det_fate=model._det_fate, zero_biomass_accum=False, default_gs=False).round(6)
# groups_df

,group_name,trophic_info,tl,ge,ee,catch,biomass,pb,qb,p,...,M0,gs,egestion,respiration,biomass_accum,emigration,immigration,net_migration,flow_to_det,detritus_import
group_seq,,,,,,,,,,,,,,,,,,,,,
33,diet_import,Import,1.000000,0.000000,1.0,0.00000,1.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
32,Detritus,DET,1.000000,1.000000,1.0,0.00000,132.610000,9.680013,9.680013,1283.666580,...,0.0,0.0,0.0,0.0,512.769167,0.0,0.0,0.0,0.0,0.0
31,Phytoplankton,PP,1.000000,1.000000,NaN,0.00000,12.950000,105.034942,105.034942,1360.202500,...,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0
30,Small zooplankton,Regular,2.102536,0.250000,NaN,0.00000,71.860000,4.815014,19.260056,346.006906,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
29,Large zooplankton,Regular,2.418964,0.131741,NaN,0.00000,14.680000,2.629147,19.956997,38.595878,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
28,Other ben. inver.,Regular,2.000000,0.300000,NaN,0.00000,5.700000,1.609362,5.364541,9.173364,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
27,Polychaetes,Regular,2.154734,0.273963,NaN,0.00000,14.820000,2.110523,7.703680,31.277944,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
26,Molluscs,Regular,2.000000,0.090000,NaN,0.03450,53.440000,0.429482,4.772024,22.951529,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
25,Echinoderms,Regular,2.000000,0.300000,NaN,0.00000,81.900000,0.410240,1.367467,33.598660,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0
